Undersampling + Sliding Window

In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # mảng index step = 1
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # mảng index step = 5
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class
        classes = np.unique(window_labels_step1)
        
        for c in classes:
            # các class hiếm dùng step =1
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            
            # các class còn lại dùng step = global step
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
            
            # đếm số lượng cửa sổ của class đó
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                c_indices = np.random.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [100]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)  # index các cửa sổ hợp lệ sau khi lọc và Undersampling
            
        # trộn đều danh sách index
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    # trả về số lượng cửa sổ hợp lệ
    def __len__(self):
        return len(self.valid_indices)

    # trả về feature và label của cửa sổ trượt tại index đã lọc và xáo trộn
    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_CLASSES = len(train_df['label'].unique())


print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=TEST_STEP_SIZE)
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=TEST_STEP_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Đang tính toán các cửa sổ hợp lệ (global step=5) và Undersampling...
Class 0: Giữ nguyên 12897 cửa sổ (step=5).
Class 1: Giảm từ 314644 xuống 20000 cửa sổ.
Class 2: Giữ nguyên 1633 cửa sổ (step=5).
Class 3: Giảm từ 23563 xuống 20000 cửa sổ.
Class 4: Giữ nguyên 12333 cửa sổ (step=5).
Class 5: Giữ nguyên 1592 cửa sổ (step=5).
Class 6: Giữ nguyên 7698 cửa sổ (step=5).
Class 7: Giảm từ 69830 xuống 20000 cửa sổ.
Class 8: Giữ nguyên 10884 cửa sổ (step=5).
Class 9: Giảm từ 26987 xuống 20000 cửa sổ.
Class 10: Giữ nguyên 12065 cửa sổ (step=5).
Tổng số cửa sổ sau khi lọc và Undersampling: 139102
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 14880 cửa sổ (step=1).
Class 1: Giữ nguyên 363049 cửa sổ (step=1).
Class 2: Giữ nguyên 1884 cửa sổ (step=1).
Class 3: Giữ nguyên 27186 cửa sổ (step=1).
Class 4: Giữ nguyên 

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# hàm tính focal loss (hiện đang không dùng Focal Loss)
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


In [5]:
import torch
import torch.nn as nn

NUM_FEATURES = train_dataset.X.shape[1]

# tính attention cho output của BiLSTM và tạo vector ngữ cảnh (context vector)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# SE Block: từ đầu ra là các channel từ CNN, kênh nào chiếm tín hiệu rõ ràng thì trọng số là 1, kênh nào bị nhiễu or sai lệch thì trọng số = 0
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # từ các channel của CNN, tính average pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        
        # tính điểm dựa trên 1 mạng neural channel => channel/8 => channel và đi qua hàm sigmoid để tạo thang điểm 0-1 cho từng kênh
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    # b,c: batch size, số kênh
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # tính toán điểm trung bình của các channel theo chiều thời gian
        y = self.fc(y).view(b, c, 1)    # tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # nhân điểm số này với các channel để khuếch đại kênh chuẩn và bóp nghẹt kênh bị nhiễm Drift

# residual block: CNN tích hợp SE Block và shortcut connection 
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding) # đi qua lớp convoluation đầu tiên
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels) # group norm: chuẩn hóa độc lập theo từng nhóm kênh
        self.relu = nn.ReLU() # hàm kích hoạt ReLU
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding) # đi qua lớp convolution thứ hai
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels) # chuẩn hóa sau lớp convolution thứ hai
        self.dropout = nn.Dropout1d(0.2)
        
        # SE Block: chấm điểm đầu ra cho các channel của CNN
        self.se = SEBlock1D(out_channels)
        
        # shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels: # nếu số kênh đầu vào khác số kênh đầu ra thì shortcut phải dùng convolution để điều chỉnh số kênh
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        out = self.se(out)
        out += residual  
        out = self.relu(out)
        return out


# model CNN-BiLSTM hoàn chỉnh
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # 2 khối Residual Block
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # cho BiLSTM nhận đầu vào là 128 channel, tạo ra output có chiều dài hiden_size*2
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # chuẩn hóa layer norm theo từng time-step
        self.layer_norm = nn.LayerNorm(hidden_size * 2)

        # tính attention score cho output của BiLSTM và tạo vector ngữ cảnh (context vector)
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x ban đầu: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) # chuyển thành (Batch, Features, SeqLen) để phù hợp với đầu vào của CNN
        
        # đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        # chuyển lại thành (batch, seq_len, features) để phù hợp với đầu vào của BiLSTM
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # chuẩn hóa layer norm theo từng time-step + tính attention score để tạo vector ngữ cảnh
        out = self.layer_norm(out)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua 2 lớp fully connected với dropout và layer norm để tăng cường ổn định
        out = self.dropout(context_vector)
        out = self.fc1(out) 
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {DEVICE}")
# khởi tạo mô hình
model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
print(model)

Đang sử dụng thiết bị: cuda
CNN_BiLSTM_Attention(
  (res1): ResidualBlock1D(
    (conv1): Conv1d(314, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (relu): ReLU()
    (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (dropout): Dropout1d(p=0.2, inplace=False)
    (se): SEBlock1D(
      (avg_pool): AdaptiveAvgPool1d(output_size=1)
      (fc): Sequential(
        (0): Linear(in_features=64, out_features=8, bias=False)
        (1): ReLU(inplace=True)
        (2): Linear(in_features=8, out_features=64, bias=False)
        (3): Sigmoid()
      )
    )
    (shortcut): Sequential(
      (0): Conv1d(314, 64, kernel_size=(1,), stride=(1,))
      (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    )
  )
  (res2): ResidualBlock1D(
    (conv1): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (relu):

In [6]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# tính lại trọng số lớp sau khi đã Undersampling để tạo class weight cho Loss Function (hiện ta không dùng)
actual_labels = []
for idx in train_dataset.valid_indices:
    label = train_dataset.y[idx + TIME_STEPS - 1].item()
    actual_labels.append(label)

actual_labels = np.array(actual_labels)

classes_in_train = np.unique(actual_labels)
new_weights = compute_class_weight(
    class_weight='balanced', 
    classes=classes_in_train, 
    y=actual_labels
)

# căn bậc 2 trọng số
smoothed_new_weights = np.sqrt(new_weights)

# khởi tạo focal loss với trọng số đã được căn bậc 2
alpha_tensor = torch.tensor(smoothed_new_weights, dtype=torch.float32).to(DEVICE)
criterion = FocalLoss(alpha=alpha_tensor, gamma=3.0)

print(f"Trọng số mới sau khi Undersampling và Smooth:\n{alpha_tensor.cpu().numpy()}")

Trọng số mới sau khi Undersampling và Smooth:
[0.990207  0.7951615 2.7827697 0.7951615 1.0125954 2.818375  1.2816852
 0.7951615 1.0778941 0.7951615 1.0237801]


In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 5

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_exper_3.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_exper_3.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds,digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.9480, Train Macro F1: 0.8168 | Val Loss: 0.6309, Val Macro F1: 0.8762


Epoch [2/30] - Train Loss: 0.7236, Train Macro F1: 0.9354 | Val Loss: 0.6041, Val Macro F1: 0.8977


Epoch [3/30] - Train Loss: 0.6932, Train Macro F1: 0.9491 | Val Loss: 0.6019, Val Macro F1: 0.8972


Epoch [4/30] - Train Loss: 0.6706, Train Macro F1: 0.9613 | Val Loss: 0.6015, Val Macro F1: 0.9127


...

Epoch [27/30] - Train Loss: 0.5955, Train Macro F1: 0.9942 | Val Loss: 0.5828, Val Macro F1: 0.9258


Epoch [28/30] - Train Loss: 0.5942, Train Macro F1: 0.9952 | Val Loss: 0.5639, Val Macro F1: 0.9468


Epoch [29/30] - Train Loss: 0.5924, Train Macro F1: 0.9955 | Val Loss: 0.5708, Val Macro F1: 0.9370


Epoch [30/30] - Train Loss: 0.5938, Train Macro F1: 0.9952 | Val Loss: 0.5775, Val Macro F1: 0.9375

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.8542    0.9704    0.9086     19846
           1     0.9999    1.0000    1.0000    484077
           2     0.4388    0.9952    0.6091      2515
           3     0.9990    0.9007    0.9473     36253
           4     0.6086    0.8548    0.7110     18979
           5     0.9984    0.9943    0.9963      2451
           6     0.7097    0.7023    0.7060     11847
           7     0.9999    0.9928    0.9963    107436
           8     0.8006    0.9901    0.8853     16746
           9     1.0000    0.6775    0.8078     41514
          10     0.9734    0.9885    0.9809     18567

    accuracy                         0.9671    760231
   macro avg     0.8530    0.9151    0.8680    760231
weighted avg     0.9749    0.9671    0.9680    760231



In [7]:
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report, roc_auc_score
from tqdm import tqdm
import numpy as np

print("\n--- ĐÁNH GIÁ CHỈ SỐ AUC-ROC TRÊN TẬP TEST ---")

# Khởi tạo và load lại trọng số tốt nhất cho model (đảm bảo model đã được định nghĩa ở trên)
NUM_FEATURES = train_dataset.X.shape[1]
NUM_CLASSES = len(train_df['label'].unique())
TIME_STEPS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
model.load_state_dict(torch.load(r"C:\Users\Admin\Downloads\IoT Dataset\model_final\best_cnn_bilstm_exper_3.pth", weights_only=True))
model.eval()

all_test_probs = []
all_test_targets = []
all_test_preds = []

# Trích xuất phân bố xác suất và nhãn thực tế
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Đang dự đoán Model]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs) # Trả về logits sinh ra từ thiết kế CNN-BiLSTM_Attention
        
        # Chuyển logits thành xác suất với Softmax
        probs = F.softmax(outputs, dim=1)
        
        # Dự đoán classes (Lấy Index có xác suất cao nhất)
        _, predicted = torch.max(probs, 1)
        
        all_test_probs.append(probs.cpu().numpy())
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

# Gộp danh sách các xác suất thành mảng numpy
all_test_probs = np.concatenate(all_test_probs, axis=0)

# Tính toán Macro OVR AUC-ROC
auc_roc = roc_auc_score(all_test_targets, all_test_probs, multi_class='ovr', average='macro')

print(f"\n=> Tỷ lệ AUC-ROC (Macro, OVR): {auc_roc:.4f}")
print("\nClassification Report:\n")
print(classification_report(all_test_targets, all_test_preds, digits=4))


--- ĐÁNH GIÁ CHỈ SỐ AUC-ROC TRÊN TẬP TEST ---



=> Tỷ lệ AUC-ROC (Macro, OVR): 0.9858

Classification Report:

              precision    recall  f1-score   support

           0     0.8542    0.9704    0.9086     19846
           1     0.9999    1.0000    1.0000    484077
           2     0.4388    0.9952    0.6091      2515
           3     0.9990    0.9007    0.9473     36253
           4     0.6086    0.8548    0.7110     18979
           5     0.9984    0.9943    0.9963      2451
           6     0.7097    0.7023    0.7060     11847
           7     0.9999    0.9928    0.9963    107436
           8     0.8006    0.9901    0.8853     16746
           9     1.0000    0.6775    0.8078     41514
          10     0.9734    0.9885    0.9809     18567

    accuracy                         0.9671    760231
   macro avg     0.8530    0.9151    0.8680    760231
weighted avg     0.9749    0.9671    0.9680    760231

